### Business Understanding & Problem Formulation

**Business Problem:**
Food systems contribute ~30% of global greenhouse gas emissions, but emissions vary dramatically by product and country. Policymakers and food companies need to identify high-impact areas for intervention.


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import plotly.io as pio

pio.renderers.default = 'notebook'



# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

# Option A: Use text without subscript (simplest)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False


# LOAD AND PREPARE DATA

In [2]:

# Load datasets
footprints = pd.read_csv('../data/raw/Food_Production.csv')
faostat = pd.read_csv('../data/raw/data_20260501_205633.csv', encoding='latin1')

# Display basic info
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

print("\n--- Footprints Dataset ---")
print(f"Shape: {footprints.shape}")
print(f"Columns: {footprints.columns.tolist()}")
print(f"Missing values per column:")
print(footprints.isnull().sum())

print("\n--- FAOSTAT Dataset (first rows) ---")
print(f"Shape: {faostat.shape}")
print(f"Columns: {faostat.columns[:10].tolist()}...")
print(f"Missing values per column:")
print(faostat.isnull().sum())

C:\Users\CMG\AppData\Local\Temp\ipykernel_11056\3583505010.py:3: DtypeWarning: Columns (11,14,17,20,23,26,29,32,35,38,41,44,47,50,53,56,59,62,65,68,71,74,77,80,83,86,89,92,95,98,101,104,107,110,113,116,119,122,125,128,131,134,137,140,143,146,149,152,155,158,161,164,167,170,173,176,179,182,185,188,191,194,197) have mixed types. Specify dtype option on import or set low_memory=False.
  faostat = pd.read_csv('../data/raw/data_20260501_205633.csv', encoding='latin1')


DATASET OVERVIEW

--- Footprints Dataset ---
Shape: (43, 23)
Columns: ['Food product', 'Land use change', 'Animal Feed', 'Farm', 'Processing', 'Transport', 'Packging', 'Retail', 'Total_emissions', 'Eutrophying emissions per 1000kcal (gPO₄eq per 1000kcal)', 'Eutrophying emissions per kilogram (gPO₄eq per kilogram)', 'Eutrophying emissions per 100g protein (gPO₄eq per 100 grams protein)', 'Freshwater withdrawals per 1000kcal (liters per 1000kcal)', 'Freshwater withdrawals per 100g protein (liters per 100g protein)', 'Freshwater withdrawals per kilogram (liters per kilogram)', 'Greenhouse gas emissions per 1000kcal (kgCO₂eq per 1000kcal)', 'Greenhouse gas emissions per 100g protein (kgCO₂eq per 100g protein)', 'Land use per 1000kcal (m² per 1000kcal)', 'Land use per kilogram (m² per kilogram)', 'Land use per 100g protein (m² per 100g protein)', 'Scarcity-weighted water use per kilogram (liters per kilogram)', 'Scarcity-weighted water use per 100g protein (liters per 100g protein)', 'Scarc

# Processing

In [3]:
def clean_faostat_data(faostat_df):
    """Reshape FAOSTAT from wide to long format"""

    # Identify year columns
    year_cols = [col for col in faostat_df.columns
                 if col.startswith('Y') and col[1:].isdigit()]

    print(f"\nFound {len(year_cols)} year columns: {year_cols[:5]}... to {year_cols[-1]}")

    # Melt to long format
    faostat_long = faostat_df.melt(
        id_vars=['Area Code', 'Area', 'Item Code', 'Item',
                 'Element Code', 'Element', 'Unit'],
        value_vars=year_cols,
        var_name='Year',
        value_name='Production'
    )

    # Clean Year column
    faostat_long['Year'] = faostat_long['Year'].str.replace('Y', '').astype(int)

    # Remove missing values
    faostat_long = faostat_long.dropna(subset=['Production'])

    # Filter for Production element (Element Code 5510)
    # Note: You may need to check your specific Element codes
    production_codes = [5510, 5312]  # Production in tonnes, Production in Mt
    faostat_prod = faostat_long[faostat_long['Element Code'].isin(production_codes)]

    # Filter for unit "tonnes" or "t"
    faostat_prod = faostat_prod[faostat_prod['Unit'].str.contains('ton|t', case=False, na=False)]

    print(f"\nFiltered FAOSTAT: {faostat_prod.shape[0]} rows")
    print(f"Years range: {faostat_prod['Year'].min()} - {faostat_prod['Year'].max()}")
    print(f"Unique countries: {faostat_prod['Area'].nunique()}")
    print(f"Unique items: {faostat_prod['Item'].nunique()}")

    return faostat_prod


def apply_product_mapping(faostat_prod, product_mapping):
    """Map FAO items to food products"""

    # Create reverse mapping
    fao_to_food = {}
    for food_product, fao_items in product_mapping.items():
        for fao_item in fao_items:
            fao_to_food[fao_item] = food_product

    # Apply mapping
    faostat_prod['Food product'] = faostat_prod['Item'].map(fao_to_food)

    # Show mapping coverage
    mapped_count = faostat_prod['Food product'].notna().sum()
    total_rows = len(faostat_prod)
    print(f"\nMapping coverage: {mapped_count}/{total_rows} rows ({mapped_count / total_rows * 100:.1f}%)")

    # Show unmapped items (top 20)
    unmapped = faostat_prod[faostat_prod['Food product'].isna()]['Item'].value_counts().head(20)
    print("\nTop 20 unmapped FAO items:")
    print(unmapped)

    # Drop unmapped rows
    faostat_mapped = faostat_prod.dropna(subset=['Food product']).copy()

    return faostat_mapped


def calculate_emissions(faostat_mapped, footprints):
    """Calculate total emissions from production volumes"""

    # Get emission factors
    emission_factors = footprints[['Food product', 'Total_emissions']].copy()
    emission_factors = emission_factors.dropna(subset=['Total_emissions'])

    # Merge
    merged = faostat_mapped.merge(emission_factors, on='Food product', how='inner')

    # Convert production to kg and calculate emissions
    merged['Production_kg'] = merged['Production'] * 1000
    merged['Emissions_kgCO2eq'] = merged['Production_kg'] * merged['Total_emissions']
    merged['Emissions_MtCO2eq'] = merged['Emissions_kgCO2eq'] / 1e9

    print(f"\nMerged dataset shape: {merged.shape}")
    print(f"Total global emissions (all years): {merged['Emissions_MtCO2eq'].sum():.2f} Mt CO₂eq")

    return merged


In [21]:
import os
os.getcwd()

'F:\\Projects\\SDS-FoodProduct\\REPO\\notebooks'

In [23]:
from product_mapped import product_mapping

In [24]:
# Execute cleaning
faostat_clean = clean_faostat_data(faostat)
faostat_mapped = apply_product_mapping(faostat_clean, product_mapping)

merged_data = calculate_emissions(faostat_mapped, footprints)


Found 63 year columns: ['Y1961', 'Y1962', 'Y1963', 'Y1964', 'Y1965']... to Y2023

Filtered FAOSTAT: 1554707 rows
Years range: 1961 - 2023
Unique countries: 245
Unique items: 280

Mapping coverage: 960659/1554707 rows (61.8%)

Top 20 unmapped FAO items:
Item
Raw hides and skins of cattle                       13435
Cattle fat, unrendered                              13435
Edible offal of cattle, fresh, chilled or frozen    13435
Oilcrops, Cake Equivalent                           13023
Edible offal of sheep, fresh, chilled or frozen     12580
Raw hides and skins of sheep or lambs               12551
Sheep fat, unrendered                               12551
Fat of pigs                                         12497
Edible offal of pigs, fresh, chilled or frozen      12497
Meat of goat, fresh or chilled                      12294
Edible offal of goat, fresh, chilled or frozen      12152
Raw hides and skins of goats or kids                12123
Goat fat, unrendered                         

In [25]:
merged_data

,Area Code,Area,Item Code,Item,Element Code,Element,Unit,Year,Production,Food product,Total_emissions,Production_kg,Emissions_kgCO2eq,Emissions_MtCO2eq
0,2,Afghanistan,221,"Almonds, in shell",5510,Production,t,1961,0.000000e+00,Nuts,0.2,0.000000e+00,0.000000e+00,0.000000
1,2,Afghanistan,515,Apples,5510,Production,t,1961,1.510000e+04,Apples,0.3,1.510000e+07,4.530000e+06,0.004530
2,2,Afghanistan,526,Apricots,5510,Production,t,1961,3.200000e+04,Other Fruit,0.7,3.200000e+07,2.240000e+07,0.022400
3,2,Afghanistan,44,Barley,5510,Production,t,1961,3.780000e+05,Barley (Beer),1.1,3.780000e+08,4.158000e+08,0.415800
4,2,Afghanistan,568,Cantaloupes and other melons,5510,Production,t,1961,1.570000e+04,Other Fruit,0.7,1.570000e+07,1.099000e+07,0.010990
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
960654,5817,Net Food Importing Developing Countries,1720,"Roots and Tubers, Total",5510,Production,t,2023,2.447746e+08,Root Vegetables,0.3,2.447746e+11,7.343237e+10,73.432368
960655,5817,Net Food Importing Developing Countries,1807,Sheep and Goat Meat,5510,Production,t,2023,4.144326e+06,Lamb & Mutton,24.5,4.144326e+09,1.015360e+11,101.535993
960656,5817,Net Food Importing Developing Countries,1723,Sugar Crops Primary,5510,Production,t,2023,2.306929e+08,Dark Chocolate,18.7,2.306929e+11,4.313958e+12,4313.957845
960657,5817,Net Food Importing Developing Countries,1729,"Treenuts, Total",5510,Production,t,2023,2.812939e+06,Nuts,0.2,2.812939e+09,5.625878e+08,0.562588


### Save the dataset

In [ ]:
import os
os.makedirs("../data/processed/", exist_ok=True)

In [27]:
merged_data.to_csv('../data/processed/integrated_food_emissions_by_country_year_product.csv', index=False)